# Traning the self built GPT2 model from scratch

In [1]:
# Config
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

num_return_sequences = 5
max_length = 30

torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Model
from nanoGPT2 import GPT, GPTConfig

# model = GPT.from_pretrained("gpt2")
model = GPT(GPTConfig())
model.eval()
model.to(device)

print(device)

/home/kevin/git_repos/dat255-teaching-assistant/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
# Load data
with open('training_data/shakespeare.txt', 'r') as f:
    text = f.read()

data = text[:1000]
print(data[:100])
# # Encode with tiktoken gpt2 bpe
# enc = tiktoken.get_encoding("gpt2")
# tokens = enc.encode(text)


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [3]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(data)
print(tokens[:24])

[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13]


In [4]:
B,T = 4,32

# This will create 4 batches of 32 tokens each (each row is a batch).
# The model will predict a token for each batch.
# x: [4, 32] is the input to the model.
# y: [4, 32] is the target to the model.
# The output/target is shifted by 1 token to the right, so the model will predict the next token in the sequence.
buf = torch.tensor(tokens[:B*T + 1])
x = buf[:-1].view(B, T).to(device)
y = buf[1:].view(B, T).to(device)
print(x[:4,:6])
print(y[:4,:6])

# Now the the expected output in y is at the same position as the input in x.
# So for example for the third index (token 25), the expected output is y's third index (token 198).

tensor([[ 5962, 22307,    25,   198,  8421,   356],
        [  477, 12939,  2138,   284,  4656,   621],
        [  385,  1526, 28599,   318,  4039,  4472],
        [  683,    11,   290,   356,  1183,   423]], device='cuda:0')
tensor([[22307,    25,   198,  8421,   356,  5120],
        [12939,  2138,   284,  4656,   621,   284],
        [ 1526, 28599,   318,  4039,  4472,   284],
        [   11,   290,   356,  1183,   423, 11676]], device='cuda:0')


In [5]:
# Testing forward pass
logits, _ = model(x)
print(logits.shape)

torch.Size([4, 32, 50257])


In [6]:
# calculate loss
logits, loss = model(x, y)
print("loss: ", loss)

# Expected loss is -ln(1/50257) (cross entropy loss for random initialization)
import math
print("expected loss: ", -math.log(1/50257))

loss:  tensor(11.0088, device='cuda:0', grad_fn=<NllLossBackward0>)
expected loss:  10.82490511970208


In [7]:
# Training on single batch of data for verification
max_iters = 50
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for iter in range(max_iters):
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"step {iter} loss: {loss.item()}")

# This test shows that we can overfit a single batch of data, which is a good sign and expected.

step 0 loss: 11.008783340454102
step 1 loss: 8.22949504852295
step 2 loss: 7.110901832580566
step 3 loss: 13.991765975952148
step 4 loss: 11.257904052734375
step 5 loss: 8.134092330932617
step 6 loss: 7.701958656311035
step 7 loss: 7.427379131317139
step 8 loss: 7.404960632324219
step 9 loss: 7.069421768188477
step 10 loss: 6.794597148895264
step 11 loss: 6.5917487144470215
step 12 loss: 6.353291988372803
step 13 loss: 6.037240028381348
step 14 loss: 5.801719665527344
step 15 loss: 5.410815238952637
step 16 loss: 5.166949272155762
step 17 loss: 4.856939792633057
step 18 loss: 4.52362585067749
step 19 loss: 4.3143157958984375
step 20 loss: 3.894552707672119
step 21 loss: 3.6466941833496094
step 22 loss: 3.426190137863159
step 23 loss: 3.1704418659210205
step 24 loss: 2.967515468597412
step 25 loss: 2.7589664459228516
step 26 loss: 2.5581352710723877
step 27 loss: 2.4080004692077637
step 28 loss: 2.2281837463378906
step 29 loss: 2.0530686378479004
step 30 loss: 1.9154677391052246
step 31

In [ ]:
# import importlib
# import nanoGPT2
# importlib.reload(nanoGPT2)  # pick up edits to nanoGPT2.py without restarting the kernel
from nanoGPT2 import DataLoaderTest

# Training loop
train_loader = DataLoaderTest(B, T)

for iter in range(max_iters):
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"step {iter} loss: {loss.item()}")


step 0 loss: 0.12522847950458527
step 1 loss: 9.862811088562012
step 2 loss: 8.113083839416504
step 3 loss: 9.198558807373047
step 4 loss: 8.51329231262207
step 5 loss: 7.966564655303955
step 6 loss: 8.777484893798828
step 7 loss: 8.296913146972656
step 8 loss: 7.9217705726623535
step 9 loss: 7.717179298400879
step 10 loss: 8.264278411865234
step 11 loss: 7.247401714324951
step 12 loss: 7.588109970092773
step 13 loss: 7.31709098815918
step 14 loss: 7.366588115692139
step 15 loss: 7.358926773071289
step 16 loss: 7.297393798828125
step 17 loss: 8.273046493530273
step 18 loss: 7.137831687927246
step 19 loss: 7.780152320861816
step 20 loss: 7.460651397705078
step 21 loss: 7.662188529968262
step 22 loss: 6.27276611328125
step 23 loss: 6.645234107971191
step 24 loss: 6.612757682800293
step 25 loss: 6.405706882476807
step 26 loss: 6.473018169403076
step 27 loss: 7.5014262199401855
step 28 loss: 7.041576385498047
step 29 loss: 6.8314971923828125
step 30 loss: 6.879134178161621
step 31 loss: 7.